# W10｜Observability / Testing / Evaluation｜教学版

**建议学习时间：5h**

这本 Notebook 的定位是“第二教材”，不是代码仓库。  
正确使用方式：

1. 先看课程视频；
2. 回到 Notebook 看“是什么 / 为什么 / 什么时候用”；
3. 带着“你应该观察什么”再运行代码；
4. 先自己做题，再展开参考答案；
5. 最后把同一机制接入 ChatBI。

## Excel 任务表 1:1 对照

| # | 模块 | Excel任务 | 输出/验收 | Notebook处理方式 |
|---:|---|---|---|---|
| 1 | Backend | 按下方课程映射完成 Testing + Tracing + Eval | 重点是可定位、可回归、可量化 | 知识点 + Checkpoint/参考答案 |
| 2 | Agent | 学 Trace + Golden Evaluation | 知道 Agent 不能只靠主观测试 | 知识点 + Checkpoint/参考答案 |
| 3 | Lab | 故意制造 1 次慢请求/失败 | 从日志定位到具体步骤 | 明确作业 + 参考答案/参考实现 |
| 4 | ChatBI | 加 trace_id + 10 条 Golden Cases | 可追踪 Intent→SQL→DB→Answer | 明确作业 + 参考答案/参考实现 |
| 5 | 复盘 | 记录 3 个质量指标 | Intent/Query/Chart 等 | 明确作业 + 参考答案/参考实现 |

> **规则**：Task 1-2 是学习任务，用 Checkpoint/面试题验收；Task 3-5 是作业任务，下面给明确参考答案/参考实现。设计题没有唯一答案时给参考图/评分标准。

## Excel 作业参考答案 / 验收标准

### Task 3｜慢/失败注入
给 DB 或 Tool step 人为 sleep / throw exception，通过相同 `trace_id` 找到最慢/失败 span。

### Task 4｜全链路
Trace 至少能串起 `Intent → SQL/Plan → DB → Answer`；Golden Cases 至少 10 条。

### Task 5｜三个质量指标参考
1. Intent/Plan Accuracy；
2. Query Execution Success / Result Accuracy；
3. Answer/Chart Appropriateness。

> 建议：先完成作业，再对照参考答案。不要直接照抄。

## 本周学习目标

        - 理解 log/metric/trace
- 建立 trace_id/span 心智模型
- 建立分层 golden evaluation

        ## 本周前置检查

        - 完成 W06-W09；已有 golden cases

        ## 本周必须记住的英文术语

        - `structured logging`：结构化日志
- `trace_id`：链路ID
- `span`：追踪片段
- `latency`：延迟
- `evaluation`：评估
- `error taxonomy`：错误分类

        > 可以听中文翻译，但这些英文术语一定要保留，因为代码、日志、文档和面试都会使用英文。

## 1. Logs / Metrics / Traces 的区别

    **是什么（What）**  
    Logs 记录离散事件；Metrics 记录可聚合数值；Trace 记录一次请求跨多个步骤的完整路径。

    **为什么需要（Why）**  
    AI 请求经常跨 LLM、Tool、DB，多步骤 latency 和错误很难靠普通日志定位。

    **什么时候用（When）**  
    Production debugging / SLO / Agent trace。

    **常见误区 / 容易踩坑**  
    只记录最终报错字符串不够；也不要把敏感 prompt/用户数据无脑写日志。


**中英术语**：`structured log` = 结构化日志；`metric` = 指标；`trace` = 链路追踪；`span` = 追踪片段；`trace_id` = 追踪ID

### 代码实验：建立最小 trace/span 模型

**这段代码想让你看什么？**

运行后找出最慢 step；以后 LangSmith/OpenTelemetry 只是更完整的实现。

> 不要先看代码细节。先带着上面的观察目标运行，再回头逐行读。

In [ ]:
import time, uuid, json
from contextlib import contextmanager

trace_id = str(uuid.uuid4())
events = []

def log_event(event, **kwargs):
    row = {"trace_id": trace_id, "event": event, **kwargs}
    events.append(row)
    print(json.dumps(row, ensure_ascii=False))

@contextmanager
def span(name):
    start = time.perf_counter()
    log_event(name + ".start")
    try:
        yield
        log_event(name + ".end", ok=True,
                  latency_ms=round((time.perf_counter()-start)*1000, 2))
    except Exception as e:
        log_event(name + ".end", ok=False, error=str(e),
                  latency_ms=round((time.perf_counter()-start)*1000, 2))
        raise

with span("intent"):
    time.sleep(0.03)

with span("database"):
    time.sleep(0.01)

ends = [e for e in events if e["event"].endswith(".end")]
print("slowest:", max(ends, key=lambda x: x["latency_ms"]))

## 2. Evaluation 要分层

    **是什么（What）**  
    把最终回答拆成 Intent、Plan、SQL、Execution、Answer 等层分别判断。

    **为什么需要（Why）**  
    只有最终 answer 评分会掩盖根因。

    **什么时候用（When）**  
    Agent/ChatBI 回归测试。

    **常见误区 / 容易踩坑**  
    不要只用 LLM-as-a-judge；能 deterministic 判断的层尽量 deterministic。


**中英术语**：`evaluation` = 评估；`error taxonomy` = 错误分类；`regression test` = 回归测试

In [ ]:
golden = [
    {"q":"Show monthly trend",
     "expected":{"intent":"trend","dimension":"month"},
     "pred":{"intent":"trend","dimension":"month"}},
    {"q":"Compare channels",
     "expected":{"intent":"comparison","dimension":"channel"},
     "pred":{"intent":"comparison","dimension":"channel"}},
    {"q":"Top use cases",
     "expected":{"intent":"ranking","dimension":"use_case"},
     "pred":{"intent":"comparison","dimension":"use_case"}},
]

def accuracy(field):
    return sum(x["expected"][field] == x["pred"][field] for x in golden) / len(golden)

print("intent acc:", accuracy("intent"))
print("dimension acc:", accuracy("dimension"))

### 练习：建立错误分类

把错误至少分成 intent_error / plan_error / sql_error / execution_error / answer_error。

<details>
    <summary><b>参考答案</b></summary>

    ```python
error_buckets = {
    "intent_error": [],
    "plan_error": [],
    "sql_error": [],
    "execution_error": [],
    "answer_error": [],
}
```

    </details>

### Checkpoint

1. 为什么 Trace 比普通日志更适合多步骤 Agent？
2. 哪些东西适合 deterministic eval？
3. 为什么 observability 不能等上线后才加？

<details>
    <summary><b>参考答案（建议先自己回答）</b></summary>

    **1.** Trace 能把同一次请求中的 LLM/tool/DB step 串起来并记录层级/耗时。

**2.** 结构化 intent、字段、SQL 执行成功、数值结果等；开放自然语言质量可结合规则和 LLM judge。

**3.** 没有 trace/metrics 后，开发阶段的错误也很难定位，且上线后补埋点成本更高。

    </details>

## ChatBI 本周作业

        ChatBI 每次请求至少记录：
`trace_id / intent latency / DB latency / total latency / success`

Golden set 增加到 10~20 条，并输出错误分类。

### 本周面试 / 理解检查

1. Logs、Metrics、Traces 的区别？
2. 为什么 Agent Eval 要分层？
3. Trace 中最重要的第一批字段？

<details>
    <summary><b>参考答案（建议先自己回答）</b></summary>

    **1.** 日志是事件，指标是聚合数值，Trace 是一次请求跨组件/步骤的完整链路。

**2.** 为了区分错误发生在理解、计划、查询、执行还是回答，否则难定位和优化。

**3.** trace_id、step/span、开始结束时间/latency、success/error、关键非敏感 metadata。

    </details>

## 本周结束前的自检

只有下面 4 项都能做到，才算真正完成：

- [ ] 我能不用课程原话，用自己的话解释核心概念；
- [ ] 我知道这周的代码实验到底在证明什么；
- [ ] 我至少独立完成了一道代码题；
- [ ] 我能说出一个“生产环境里如果做错会发生什么”的例子。